# 루프와 동적 라우팅

이 노트북에서 다루는 것:

1. **단순 루프** — 조건이 만족될 때까지 노드 반복
2. **`recursion_limit` 와 `GraphRecursionError`** — 무한 루프 방어
3. **더 복잡한 루프** — `a → b → {c, d} → a` 같은 다중 노드 합류 + 루프, `Literal` 라우팅 힌트
4. **사용자 입력 루프** — `input()` 으로 사람과 대화하며 종료 조건 결정

## 1. 단순 루프

```
START → a ─┬─→ END    (조건 만족)
           └─→ b → a   (조건 미만족, 다시 a 로)
```

라우터가 다음 노드 이름을 반환하거나, `END` 를 반환해 종료한다.
여러 노드가 같은 필드(`aggregate`)에 누적 갱신하므로 `operator.add` reducer 가 필요하다.

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    aggregate: Annotated[list, operator.add]

def a(state: State):
    print(f'Node A 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["A"]}

def b(state: State):
    print(f'Node B 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["B"]}

def route(state: State):
    # aggregate 길이가 7 미만이면 다시 b 로, 아니면 END
    return "b" if len(state["aggregate"]) < 7 else END

gb = StateGraph(State)
gb.add_node(a)
gb.add_node(b)
gb.add_edge(START, "a")
gb.add_conditional_edges("a", route)
gb.add_edge("b", "a")     # b 가 끝나면 다시 a 로 (루프)

graph = gb.compile()
graph.invoke({"aggregate": []})

## 2. `recursion_limit` — 무한 루프 방어

LangGraph 는 노드 실행 횟수 상한(기본 25) 을 두고 있다.
초과하면 `GraphRecursionError`. 안전장치이자, **종료 조건이 영원히 만족되지 않는 버그를 빠르게 잡는 수단**이다.

`config={"recursion_limit": N}` 으로 호출 단위로 바꿀 수 있다.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    graph.invoke({"aggregate": []}, config={"recursion_limit": 4})
except GraphRecursionError:
    print("Recursion Error: 종료 조건 도달 전에 상한에 걸림")

## 3. 더 복잡한 루프 — 분기 후 합류, 그리고 다시 루프

```
                    ┌── c ──┐
START → a ─→ b ─────┤        ├─→ a   (조건 미만족 시 다시)
         │          └── d ──┘
         └─→ END  (조건 만족)
```

두 가지 새 기법:
1. **`add_edge(["c", "d"], "a")`** — 여러 노드가 모두 끝난 뒤 한 노드로 합류
2. **`Literal["b", END]`** 반환 타입 힌트 — 라우터가 갈 수 있는 곳을 명시해 시각화/검증 도움

In [ ]:
from typing import Literal

class State2(TypedDict):
    aggregate: Annotated[list, operator.add]

def na(state: State2):
    print(f'Node A 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["A"]}

def nb(state: State2):
    print(f'Node B 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["B"]}

def nc(state: State2):
    print(f'Node C 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["C"]}

def nd(state: State2):
    print(f'Node D 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["D"]}

def route2(state: State2) -> Literal["nb", "__end__"]:
    return "nb" if len(state["aggregate"]) < 7 else END  # type: ignore[return-value]

gb = StateGraph(State2)
gb.add_node("na", na); gb.add_node("nb", nb)
gb.add_node("nc", nc); gb.add_node("nd", nd)

gb.add_edge(START, "na")
gb.add_conditional_edges("na", route2)
gb.add_edge("nb", "nc")
gb.add_edge("nb", "nd")
gb.add_edge(["nc", "nd"], "na")    # nc 와 nd 모두 끝나야 na 재실행

graph2 = gb.compile()
graph2.invoke({"aggregate": []})

In [ ]:
print(graph2.get_graph().draw_mermaid())

## 4. 사용자 입력에 따른 반복

노드 안에서 `input()` 을 호출해 **사람과 대화하며 종료 조건을 판단**할 수 있다.
여기서는 사용자가 입력에 "반복" 이라는 단어를 포함하면 한 번 더, 아니면 종료한다.

또한 **메시지 종류별로 따로 누적**할 수도 있다 — `human_messages` 와 `ai_messages` 를 분리해서 가지고 다닌다 (각각 `add_messages` reducer 적용).

> **실행 주의**: 이 셀은 `input()` 을 호출하므로 Jupyter 에서 셀을 실행한 뒤 입력 박스에 텍스트를 넣어줘야 한다.
> "반복" 이라고 치면 한 번 더, 다른 텍스트면 종료.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

class ChatLoopState(TypedDict):
    human_messages: Annotated[list[HumanMessage], add_messages]
    ai_messages: Annotated[list[AIMessage], add_messages]
    retry_num: int

def chatbot(state: ChatLoopState):
    retry_num = state["retry_num"]
    user_input = input(f"(현재 {retry_num}번째 답변) 사용자 입력: ")
    ai_message = AIMessage(f"{retry_num}번째 답변중!")
    return {
        "human_messages": [HumanMessage(content=user_input)],
        "ai_messages": [ai_message],
    }

def retry(state: ChatLoopState):
    return {"retry_num": state["retry_num"] + 1}

def route_chat(state: ChatLoopState):
    return "retry" if "반복" in state["human_messages"][-1].content else END

gb = StateGraph(ChatLoopState)
gb.add_node("chatbot", chatbot)
gb.add_node("retry", retry)
gb.add_edge(START, "chatbot")
gb.add_conditional_edges("chatbot", route_chat)
gb.add_edge("retry", "chatbot")

chat_graph = gb.compile()
print(chat_graph.get_graph().draw_mermaid())

실행 — `stream(stream_mode="updates")` 로 매 단계를 확인하며 진행해보자.

In [ ]:
for chunk in chat_graph.stream(
    {"human_messages": [], "ai_messages": [], "retry_num": 0},
    stream_mode="updates",
):
    for node, update in chunk.items():
        print(f"[{node}] {update}")

## 정리

- **루프** = 조건부 엣지가 이전(또는 자기 자신) 노드로 다시 향하면 됨
- **`recursion_limit`** = 무한 루프 방어. 호출 단위로 조정 가능. 초과 시 `GraphRecursionError`
- **`add_edge([n1, n2], target)`** = 다중 노드 합류. 모두 끝나야 target 실행
- **`Literal[...]`** 라우터 반환 타입 힌트로 가능한 분기를 표현 (시각화/정적 분석)
- **사용자 입력 루프** = 노드 안의 `input()` + 메시지 누적 + 라우터 조합

여기까지가 LangGraph 의 그래프 메커니즘 "기초" 다.
다음 단계(`basics` 폴더 밖)에서는 노드 안에 **실제 LLM 호출** 을 넣어 본격적인 에이전트를 만든다 — 그때부터 `.env` 의 API 키가 필요해진다.